# Old mapping data on HVBMG and HVVMG nrclib and genes were moved into /mnt/hdd1/clee/hdd_scratch/

# H. vuglaris and Bacteria (HVBMT) Metatranscriptome Analysis

This pipeline uses the account: clee@immunoviromics-Z10PE-D16-WS

## 0. Data Preprocessing: Metadata

In [ ]:
mkdir -p HVBMT_project/meta
cd HVBMT_project/meta
mkdir ntbk result

# Make metadata spreadsheet and import to HVBMT_project/result
# **********************exclude (-) controls from the metadata file**********************

# Check file format, ^I for tab, $ for Linux newline, ^M$ for Windows carriage return, ^M for Mac newline
cat -A result/HVBMT_metadata.txt

# check number of lines (expecting 57 lines for 57 samples)
cat -A result/HVBMT_metadata.txt | wc -l

# (OPTIONAL) Trim trailing whitespace 
sed -i 's/[ \t]*$//' result/HVBMT_metadata.txt

# Re-check file format
cat -A result/HVBMT_metadata.txt

# Ensure correct number of columns
awk -F'\t' '{print NF}' result/HVBMT_metadata.txt | sort -nu # expecting 4 columns

# Generate metadata based on sample files, can filter subsets, e.g., starting with HVBMT
ls /mnt/hdd1/sequence_data/250613_DTSA1106_1107_1108_1109_1110_1111_NovaX25B/JGCL_HVMT/|grep '_R1'|cut -f1 -d '_'|cut -f 2 -d '/'| sed '1 i SampleID' \
> result/metadata_ID.txt

# Metadata details optimization
# Convert Windows carriage return to Linux newline, remove spaces
sed -i 's/\r//;s/ //g' result/metadata_ID.txt

# delete last lines
sed -i '$d' result/metadata_ID.txt

# visually check
cat -A result/metadata_ID.txt

# check number of lines (expecting 58)
cat -A result/metadata_ID.txt | wc -l





## 1. Data Pre-processing: Quality control, Filtering, Host Removal, rRNA removal

### 1.1 FastP quality control (~ 24 hours)

In [ ]:
# Create directories
mkdir -p /mnt/hdd2/sequence_data/250613_DTSA1106_1107_1108_1109_1110_1111_NovaX25B/JGCL_HVMT/fastp

# within  /home/clee/HVBMT_project/meta
mkdir -p  result/qc

cp result/metadata_ID.txt /mnt/hdd2/sequence_data/250613_DTSA1106_1107_1108_1109_1110_1111_NovaX25B/JGCL_HVMT/

# This code will read the sampleIDs listed in metadata.txt for R1 and R2 and perform fastp on it and output results into fastp/
# Parallel quality control for multiple samples, this step requires 5x the original data space
# -j 2: indicates processing 2 samples simultaneously; j3,18s,8m; currently for 6 samples j2, 2m4.247s

# since storage is going to be an issue, I moved the entire raw read directory for HVBMT to /mnt/hdd2/sequence_data_temp

# check for files to renamed:
for file in *_S*_L008_R1_001.fastq.gz *_S*_L008_R2_001.fastq.gz; do
    prefix=$(echo $file | sed -E 's/([A-Za-z0-9]+)_S.*L008_R[12]_001.fastq.gz/\1/')
    read_type=$(echo $file | grep -o 'R[12]')
    echo "$file will be renamed to ${prefix}_${read_type}.fastq.gz"
done

# run this to change names of raw reads:
for file in *_S*_L008_R1_001.fastq.gz *_S*_L008_R2_001.fastq.gz; do
    # Extract the prefix (e.g., HVBMT39 from HVBMT39_S38_L008_R1_001.fastq.gz)
    prefix=$(echo $file | sed -E 's/([A-Za-z0-9]+)_S.*L008_R[12]_001.fastq.gz/\1/')
    
    # Extract read type (R1 or R2)
    read_type=$(echo $file | grep -o 'R[12]')  
    
    # Rename the file
    mv "$file" "${prefix}_${read_type}.fastq.gz"
done

# run this to perform quality filtering and adatpor trimming
time tail -n+2 metadata_ID.txt|cut -f1|rush -j 2 \
  "fastp -i {1}_R1.fastq.gz -I {1}_R2.fastq.gz \
    -j fastp/{1}_fastp.json -h fastp/{1}_fastp.html \
    -o fastp/{1}_1.fastq -O fastp/{1}_2.fastq \
    > fastp/{1}.log 2>&1"


# mv fastp output directory to /mnt/hdd2/
mv fastp/ /mnt/hdd2

# mv raw reads back to /mnt/hdd1/ to save space
mv 250613_DTSA1106_1107_1108_1109_1110_1111_NovaX25B /mnt/hdd1/sequence_data

# Use this script to obtain a summary of quality control results from fastp without needing to look at each file
# Summary of quality control results
echo -e "SampleID\tRaw\tClean" > temp/fastp
for i in `tail -n+2 result/metadata_ID.txt|cut -f1`; do
    echo -e -n "$i\t" >> temp/fastp
    grep 'total reads' fastp/${i}.log|uniq|cut -f2 -d ':'|tr '\n' '\t' >> temp/fastp
    echo "" >> temp/fastp
done
sed -i 's/ //g;s/\t$//' temp/fastp

# and this awk command sorts the output summary using the metadata.txt file so it's easier to read the samples
# Sort by metadata
awk 'BEGIN{FS=OFS="\t"}NR==FNR{a[$1]=$0}NR>FNR{print a[$1]}' temp/fastp result/metadata_ID.txt \
  > result/qc/fastp.txt
cat result/qc/fastp.txt


### 1.2 KneadData for Host removal (~4 days on HDDs)

In [ ]:
# KneadData is primarily dependent on Bowtie2 for host alignment, then filters out non-host sequences for downstream analysis.

# Create directories, activate environment, and record version
mkdir -p temp/hr
conda activate kneaddata
kneaddata --version # 0.12.0

# copy metadata_ID.txt to /mnt/hdd2

# copy the run_kneaddata.sh from the /home/clee/project/meta/scripts directory to /mnt/hdd2 (outside sequence folder containing raw reads and fastp filtered reads)
mv scripts/run_kneaddata.sh /mnt/hdd2

# modify path files inside shell script: run_kneaddata.sh, 
    # pointing to correct fastp filtered files
    # hardcoded database location: -db /home/clee/db/kneaddata/hvul105-bt2-db/hvul105_bt2
    # and set threads to 10 (-t 10)

# i.e.
# -db /home/clee/db/kneaddata/hvul105-bt2-db/hvul105_bt2
# sed "1~4 s/ 1:/.1:/;1~4 s/\$/\/1/" fastp/${sample}_1.fastq > /tmp/${sample}_1.fastq

# Parallel host removal for multiple samples, this step requires 5x the original data space, j5p3,18s3h; j3p16,18s1h

# ************************************************************************
# ********** MUST KEEP TERMINAL OPEN TO PREVENT JOB TERMINATION **********
# ************************************************************************
nohup tail -n+2 metadata_ID.txt \
| cut -f1 \
| rush -j 2 ./run_kneaddata.sh {} \
> kneaddata_run.log 2>&1 &

In [ ]:
# Check outputs of kneaddata
# Check file sizes, * matches any number of characters, ? matches any one character
ls -shtr temp/hr/*_paired_?.fastq 
ls -shtr temp/hr/*_paired_?.fastq | wc -l # expect 114 lines (57 samples and 2 pairs each)

# Simplify renaming
# Ubuntu system rename for Bacteria & Viral RNA
rename 's/paired_//' temp/hr/*.fastq

# Simplify renaming
# Ubuntu system rename for Host RNA
rename 's/_hvul105_bt2_bowtie2_contam_/_host_/' temp/hr/*.fastq

# knead can create a table of read counts for all samples (only decontaminated reads)
# Quality control result summary
kneaddata_read_count_table --input temp/hr \
  --output temp/kneaddata.txt

# Filter key result columns
cut -f 1,2,5,6 temp/kneaddata.txt | sed 's/_1_kneaddata//' > result/qc/sum.txt

# View the table in alignment format
csvtk -t pretty result/qc/sum.txt

# Check if IDs are paired
# paste <(head -n40 temp/hr/`tail -n+2 result/metadata_ID.txt|cut -f1|head -n1`_1.fastq|grep @) <(head -n40 temp/hr/`tail -n+2 result/metadata_ID.txt|cut -f1|head -n1`_2.fastq|grep @)
paste \
>  <(head -n40 temp/hr/$(tail -n+2 result/metadata_ID.txt | cut -f1 | head -n1)_1.fastq | sed -n '1~4p') \
>  <(head -n40 temp/hr/$(tail -n+2 result/metadata_ID.txt | cut -f1 | head -n1)_2.fastq | sed -n '1~4p')

# to get a similar table for host reads:
# run this within the temp/hr/ directory
(
  echo -e "SampleID\tContamSequences"
  grep "Total contaminate sequences in file" *.log \
    | grep "paired_contam_1.fastq" \
    | sed -E 's/.*\/(HVBMT[0-9]+)_hvul105_bt2_bowtie2_paired_contam_1\.fastq.*: ([0-9.]+).*/\1\t\2/' \
    | sort -u
) > contam_counts.txt

# move contam_counts.txt to results

# ************************************************************************
# ********** OPTIONAL **********
# ************************************************************************
# Clean up large files, high host content samples can save >90% space
# Use the absolute path of the command to ensure commands with no parameters are used; admins may use alias to customize commands with parameters, which can affect operation results
/bin/rm -rf temp/hr/*unmatched* temp/hr/reformatted* temp/hr/_temp*

# Within the temp/hr/ directory
# I only deleted the *unmatched* files


ls -l temp/hr/
# After confirming the host removal results, intermediate Fastp QC files can be deleted
# rm temp/qc/*.fastq

# I moved the intermediate fastp QC files to /mnt/hdd2/clee/bacteriome/qc/fastp

### 1.3 Ribodetector for removal of rRNA sequences

### Installing Ribodetector and dependencies

In [ ]:
# follow instrcutions here on the github page:
# https://github.com/hzi-bifo/RiboDetector?tab=readme-ov-file

# install ribodetector in conda env
conda create -n ribodetector -y

# install python version 3.8
conda install python=3.8 -y

# to correctly install pytorch to use GPU acceleration check the link here: https://pytorch.org/get-started/locally/ 
# check which version of CUDA you have
nvcc --version 
# OUTPUT
# nvcc: NVIDIA (R) Cuda compiler driver
# Copyright (c) 2005-2019 NVIDIA Corporation
# Built on Sun_Jul_28_19:07:16_PDT_2019
# Cuda compilation tools, release 10.1, V10.1.243

# since ribodetector has been tested on pytorch version 1.7.1 we'll stick with this version that also matches our CUDA version:
# link to website is here: https://pytorch.org/get-started/previous-versions/ 
conda install pytorch==1.7.1 torchvision==0.8.2 torchaudio==0.7.2 cudatoolkit=10.1 -c pytorch -y

# check if pytorch is working by first calling python3
python3

# then inside python interpretor, we will test if pytorch is working
import torch
x = torch.rand(5, 3)
print(x)

# expected OUTPUT something similar to:
tensor([[0.9816, 0.9816, 0.6903],                                                                                                                                                         
        [0.2932, 0.3861, 0.3069],                                                                                                                                                         
        [0.1112, 0.7637, 0.9058],                                                                                                                                                         
        [0.8335, 0.9191, 0.1461],                                                                                                                                                         
        [0.2255, 0.9380, 0.2298]]) 

# Additionally, to check if your GPU driver and CUDA/ROCm is enabled and accessible by PyTorch, run the following commands to return 
# whether or not the GPU driver is enabled:
# inside python3 interpretor:
import torch
torch.cuda.is_available() # should return TRUE if FALSE, then I don't know... We'll cross that bridge when we get there...
quit()

# after all checks passed, then install RiboDetector
conda install -c bioconda ribodetector -y


In [ ]:
# testing ribodetector
ribodetector -t 20 \
  -l 150 \
  -i /mnt/hdd2/clee/transcriptome/temp/hr/HVBMT01_1.fastq /mnt/hdd2/clee/transcriptome/temp/hr/HVBMT01_2.fastq \
  -m 7 \
  -e rrna \
  -o /mnt/hdd2/clee/transcriptome/temp/rrna_r/HVBMT01_1.nonrrna.fastq /mnt/hdd2/clee/transcriptome/temp/rrna_r/HVBMT01_2.nonrrna.fastq
  --log /mnt/hdd2/clee/transcriptome/temp/rrna_r/HVBMT01_ribodetector.log

# run seems successful

In [ ]:
# make a script to automate rRNA removal for all samples
#!/bin/bash
for fq1 in /mnt/hdd2/clee/transcriptome/temp/hr/HVBMT??_1.fastq; do
    sample_name=$(basename "$fq1" | cut -d'_' -f1)
    fq2="/mnt/hdd2/clee/transcriptome/temp/hr/${sample_name}_2.fastq"
    out_dir="/mnt/hdd2/clee/transcriptome/temp/hr/rrna_r"
    
    ribodetector -t 20 \
    -l 150 \
    -i "$fq1" "$fq2" \
    -m 7 \
    -e rrna \
    -o "${out_dir}/${sample_name}_1.nonrrna.fastq" "${out_dir}/${sample_name}_2.nonrrna.fastq"
    --log "${out_dir}/${sample_name}_ribo.log"
    
done

# save it to remove_rrna.sh in /mnt/hdd2/clee/transcriptome/scripts
touch remove_rrna.sh
nano remove_rrna.sh
# copy the script into the file we just made
# change its permission
chmod 777 remove_rrna.sh

# run the script
./remove_rrna.sh

# Completed all samples until sample HVBMT47
# Ran into error: CUDA error: out of memory

# Retry run with lower memory bringing down from 7 Gb to 6 Gb limit
# Run is ongoing so I'll stick with this value
# Run finished successfully!

In [ ]:
# to get quick statistics do the following (ensure that all log files from different runs are contatenated together):
echo "total_seqs,non-rrna_seq" > ribodetector_stats.csv
paste -d, ../metadata_ID.txt ribodetector_stats.csv > ribodetector_stats_with_ID.csv
rm ribodetector_stats.csv
mv ribodetector_stats_with_ID.csv ribodetector_stats.csv

# this part will extract the total number of RNA sequences given to ribodetector (described in the log file), and strip out any ANSI characters which may affect the numeric values
# then it will append it to the 2nd column of the csv file
awk -F, 'NR==FNR {tot[NR]=$1; next}
         FNR==NR {next}
         NR==FNR+ARGC-2 {non[NR-FNR]=$1; next}
         FNR==1 {print; next}
         {print $1 "," tot[FNR-1] "," non[FNR-1]}' \
    <(cat remove_rrna-1.log | grep "sequences loaded!" | cut -d" " -f6 | sed 's/\x1b\[[0-9;]*m//g') \
    ribodetector_stats.csv > ribodetector_stats_filled.csv

# this code will extract the non-rRNA sequences and append it to the 3rd column of the csv file
awk -F, 'NR==FNR {a[NR]=$1; next} NR==1 {print $0 ",non-rrna_seq"; next} NR>1 { $3=a[FNR-1]; print $1 "," $2 "," $3 }' \
    <(cat remove_rrna-1.log | grep -P 'Detected.*non-rRNA sequences' | cut -d" " -f7 | sed 's/\x1b\[[0-9;]*m//g') \
    ribodetector_stats_filled.csv > ribodetector_stats_filled_final.csv


# cleanup
rm ribodetector_stats.csv ribodetector_stats_filled.csv ribodetector_stats_final.csv

# use ribodetector_stats_filled_final.csv for graphs for view in Excel or make edits (the code above has a bug that deletes the column headers so 
# you'll need to fix it in Excel)


In [ ]:
###############################
###############################
###############################
###############################

PROCEED TO SECTION 2.0 Redo Assembly separating Noninduced & Induced samples for the final method used


###############################
###############################
###############################
###############################

## 2.0 Redo Assembly separating Noninduced & Induced samples

In [ ]:
# I will be using SqueezeMeta pipeline for assembly and annotation

In [ ]:
# this was NOT used

#!/bin/bash
#SBATCH --nodes=1
#SBATCH --partition=cenvalarc.compute
#SBATCH --mem=0
#SBATCH --time=3-00:00:00
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=64
#SBATCH --output=rna_assembly_noninduced.log
#SBATCH --job-name=rna_assembly
#SBATCH --mail-user=clee289@ucmerced.edu
#SBATCH --mail-type=ALL
#SBATCH --export=ALL

source activate megahit

time megahit -t 64 \
    -1 `tail -n+2 HVBMT_metadata_noninduced.csv | cut -f1 -d',' | sed 's/^/rrna_r\//;s/$/_1.nonrrna.fastq/'|tr '\n' ','|sed 's/,$//'` \
    -2 `tail -n+2 HVBMT_metadata_noninduced.csv | cut -f1 -d',' | sed 's/^/rrna_r\//;s/$/_2.nonrrna.fastq/'|tr '\n' ','|sed 's/,$//'` \
    -o megahit

# took ~ 25 hours (1 day)

## 2.1 Install SqueezeMeta for bacterial metatranscriptome analysis

In [ ]:
# Create package cache and environment directories on /scratch (or /data)
mkdir -p /home/clee289/borgstore/conda/conda_pkgs
mkdir -p /home/clee289/borgstore/conda/conda_envs

conda config --add pkgs_dirs /home/clee289/borgstore/conda/conda_pkgs
conda config --add envs_dirs /home/clee289/borgstore/conda/conda_envs

conda install -n base conda-libmamba-solver
conda config --set solver libmamba
conda create -n squeezemeta -c conda-forge -c bioconda -c fpusan squeezemeta=1.8 --no-channel-priority --override-channels -y

conda activate squeezemeta

# test all required dependencies are installed
./conda/conda_envs/squeezemeta/SqueezeMeta/utils/install_utils/test_install.pl

# download preformatted databases used by SqueezeMeta
#!/bin/bash
#SBATCH --nodes=1
#SBATCH --partition=cenvalarc.compute
#SBATCH --mem=0
#SBATCH --time=3-00:00:00
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=64
#SBATCH --output=sqzmta_db.log
#SBATCH --job-name=sqzmta_dbdl
#SBATCH --mail-user=clee289@ucmerced.edu
#SBATCH --mail-type=ALL
#SBATCH --export=ALL

conda activate squeezemeta

mkdir -p /home/clee289/borgstore/databases/squeezemeta_db

./conda/conda_envs/squeezemeta/SqueezeMeta/utils/install_utils/download_databases.pl /home/clee289/borgstore/databases/squeezemeta_db


In [ ]:
# this code block will use SqueezeMeta's download_databases.pl script to download it's required preformatted databases

#!/bin/bash
#SBATCH --nodes=1
#SBATCH --partition=cenvalarc.compute
#SBATCH --mem=0
#SBATCH --time=3-00:00:00
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=64
#SBATCH --output=/home/clee289/borgstore/databases/sqzmta_db_%j.log
#SBATCH --job-name=sqzmta_dbdl
#SBATCH --mail-user=clee289@ucmerced.edu
#SBATCH --mail-type=ALL

# 1. Suppress noisy wget progress gauges across all child processes
export WGETRC=/tmp/wgetrc_${SLURM_JOB_ID}
echo "progress = dot:giga" > "$WGETRC"

# 2. Cleanly initialize and activate conda in batch subshell
source /home/clee289/miniconda3/etc/profile.d/conda.sh
conda activate squeezemeta

DB_DIR="/home/clee289/borgstore/databases/squeezemeta_db"
SCRIPT_PATH="/home/clee289/borgstore/conda/conda_envs/squeezemeta/SqueezeMeta/utils/install_utils/download_databases.pl"

mkdir -p "$DB_DIR"

echo "=== [$(date)] Starting SqueezeMeta database download ==="
echo "Target directory: $DB_DIR"

# 3. Execute download using an absolute path (avoids working-directory issues)
perl "$SCRIPT_PATH" "$DB_DIR"

STATUS=$?

# 4. Clean up temporary wget configuration
rm -f "$WGETRC"

echo "=== [$(date)] Download process exited with status: $STATUS ==="

# 5. Verification step: list downloaded databases and check total size
if [ $STATUS -eq 0 ]; then
    echo "=== Downloaded directory contents and sizes: ==="
    du -sh "$DB_DIR"/*
    echo "=== Running SqueezeMeta configuration check ==="
    # Checks if SqueezeMeta detects all required database files
    test_install.pl
else
    echo "ERROR: download_databases.pl failed or terminated early." >&2
    exit $STATUS
fi

In [ ]:
# after running the block above, test all required dependencies are installed
./conda/conda_envs/squeezemeta/SqueezeMeta/utils/install_utils/test_install.pl

In [ ]:
# format input sample sheet according to SqueezeMeta's required format

# separate the metatranscriptome samples by noninduced and induced
cd /home/clee289/borgstore/HVBMT_analysis/

# create sample sheet for noninduced samples
head -n 1 HVBMT_metadata_combined_updated.csv > HVBMT_metadata_noninduced.csv
grep noninduced HVBMT_metadata_combined_updated.csv >> HVBMT_metadata_noninduced.csv

# create sample sheet for induced samples
head -n 1 HVBMT_metadata_combined_updated.csv > HVBMT_metadata_induced.csv
head -n 4 HVBMT_metadata_combined_updated.csv | tail -n 3 >> HVBMT_metadata_induced.csv
grep -v noninduced HVBMT_metadata_combined_updated.csv | tail -n+2 >> HVBMT_metadata_induced.csv

cd /home/clee289/borgstore/HVBMT_analysis

# Set path to reads directory
READ_DIR="/home/clee289/borgstore/HVBMT_analysis/temp/rrna_r"

# Generate SqueezeMeta sheet for induced samples
awk -F',' 'NR>1 && $1 != "" {
    print $1 "\t" $1 "_1.nonrrna.fastq\tpair1";
    print $1 "\t" $1 "_2.nonrrna.fastq\tpair2"
}' HVBMT_metadata_induced.csv > squeezemeta_samples_induced.txt

# Generate SqueezeMeta sheet for noninduced samples
awk -F',' 'NR>1 && $1 != "" {
    print $1 "\t" $1 "_1.nonrrna.fastq\tpair1";
    print $1 "\t" $1 "_2.nonrrna.fastq\tpair2"
}' HVBMT_metadata_noninduced.csv > squeezemeta_samples_noninduced.txt



SRC_DIR="/home/clee289/borgstore/HVBMT_analysis/temp/rrna_r"
NONIND_DIR="/home/clee289/borgstore/HVBMT_analysis/temp/rrna_r/reads_noninduced"
IND_DIR="/home/clee289/borgstore/HVBMT_analysis/temp/rrna_r/reads_induced"

# 2. Create the destination directories
mkdir -p "$NONIND_DIR" "$IND_DIR"
# move non-induced reads
awk '{print $2}' squeezemeta_samples_noninduced.txt | sort -u | while read -r file; do
    mv "$SRC_DIR/$file" "$NONIND_DIR/"
done

# move induced reads (will need to move T0 samples to this directory after completing run for noninduced samples)
awk '{print $2}' squeezemeta_samples_induced.txt | sort -u | while read -r file; do
    mv "$SRC_DIR/$file" "$IND_DIR/"
done

In [ ]:
# execute a dry run of SqueezeMeta to ensure that it works

#!/bin/bash
#SBATCH --nodes=1
#SBATCH --partition=cenvalarc.compute
#SBATCH --mem=0
#SBATCH --time=3-00:00:00
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=64
#SBATCH --output=/home/clee289/borgstore/HVBMT_analysis/sqm_noninduced_%j.log
#SBATCH --job-name=sqzm
#SBATCH --mail-user=clee289@ucmerced.edu
#SBATCH --mail-type=ALL


SqueezeMeta.pl -m coassembly \
    -p /home/clee289/borgstore/HVBMT_analysis/result/sqm_noninduced \
    -s /home/clee289/borgstore/HVBMT_analysis/squeezemeta_samples_noninduced.txt \
    -f /home/clee289/borgstore/HVBMT_analysis/temp/rrna_r/reads_noninduced \
    -a megahit \
    -t 64 \
    --empty


In [ ]:
# final executable run to generate SqueezeMeta results
# run took ~1 day, 13 hours, 11 minutues

#!/bin/bash
#SBATCH --nodes=1
#SBATCH --partition=cenvalarc.compute
#SBATCH --mem=0
#SBATCH --time=3-00:00:00
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=64
#SBATCH --output=/home/clee289/borgstore/HVBMT_analysis/result/sqm_noninduced_%j.log
#SBATCH --job-name=sqzm_nonind
#SBATCH --mail-user=clee289@ucmerced.edu
#SBATCH --mail-type=ALL

# Activate conda environment
source $(conda info --base)/etc/profile.d/conda.sh
conda activate squeezemeta

# Ensure destination directory is clean if rerunning
PROJECT_DIR="/home/clee289/borgstore/HVBMT_analysis/result/sqm_noninduced"
rm -rf "$PROJECT_DIR"

SqueezeMeta.pl -m coassembly \
    -p "$PROJECT_DIR" \
    -s /home/clee289/borgstore/HVBMT_analysis/squeezemeta_samples_noninduced.txt \
    -f /home/clee289/borgstore/HVBMT_analysis/temp/rrna_r/reads_noninduced \
    -a megahit \
    -t 64


# once finish, upload to local workstation to save results
tmux new -s squeezemeta

# in this session run rsync to transfer the large ~653 Gb output directory
rsync -avhP --info=progress2 clee289@login.rc.ucmerced.edu:/home/clee289/borgstore/HVBMT_analysis/result/sqm_noninduced /mnt/hdd3/

In [ ]:
# Analyzing SqueezeMeta results in your desktop computer

# Allocate test partition
#----------------------------------------------
# ALLOCATING RESOURCES AND TEST PARTITION (BEGIN) 
#----------------------------------------------
# allocate gpu on test partition for 60 min
salloc --partition=test --nodes=1 --gres=gpu:1 --time=01:00:00

# To stout:
salloc: Granted job allocation 347501
salloc: Nodes gnode009 are ready for job


#   replace jobid with job allocation 
srun --jobid=[jobid] --pty /bin/bash 
srun --jobid=347501 --pty /bin/bash

# if needed
# to exit out of test partition with gpu allocation
exit

# to exit of out of job
# (base) [clee289@rclogin03 hvvmg_iphop]$ exit
exit

conda activate squeezemeta
mkdir -p /home/clee289/borgstore/HVBMT_analysis/result/sqm_noninduced_analysis
sqm2zip.py /home/clee289/borgstore/HVBMT_analysis/result/sqm_noninduced /home/clee289/borgstore/HVBMT_analysis/result/sqm_noninduced_analysis

# transfert the outputted zip file to workstation
located here: /mnt/hdd3/sqm_noninduced/sqm_noninduced.zip

## 3.0 SqueeMeta Output Analysis (Noninduced samples only)

In [ ]:
# this work was performed on the lab's local workstation

# setting up R for using SQMtools in base R

# Open up a terminal on VS code connected workstation server
# execute the following in bash command line:
sudo apt update
sudo apt install -y \
    libfontconfig1-dev \
    libfreetype6-dev \
    libharfbuzz-dev \
    libfribidi-dev \
    libpng-dev \
    libtiff-dev \
    libjpeg-dev

sudo apt-get update && sudo apt-get install -y libxml2-dev    

R -e "install.packages(c('rmarkdown', 'knitr', 'languageserver', 'httpgd', 'IRkernel'), repos='https://cloud.r-project.org')"

# install SQMtools
R -e "install.packages('pathview', repos = c('https://bioconductor.org/packages/3.22/bioc', 'https://cloud.r-project.org'), verbose = TRUE)"
R -e "install.packages('SQMtools', repos = 'https://cloud.r-project.org')"

# check installation works
R -e "library(SQMtools)"



In [ ]:
# NOTE: I installed all the R packages for analysis for the metatranscriptome on the system level (/usr/bin/R)
    #   jupyter and jupyter notebook are installed on base conda environment ((base) clee@immunoviromics-Z10PE-D16-WS))

# when working on the r notebook do the following to prevent long r jobs from canceling when closing your computer
tmux new -s jupyter_session
# start a jupyter kernel
jupyter notebook --no-browser --port=8888
# copy the outputted token
http://localhost:8888/tree?token=98558525bcfbd93148884b11b5d17014e8eb3bb5f01b6168

# detach from tmux session (Ctrl B, followed by 'D' key stroke)

# Connect VS Code to the Running Server
    # Open your .ipynb notebook in VS Code.
    # In the top-right corner, click on the Kernel Selector (where it says R).
    # Select Select Another Kernel... --> Existing Jupyter Server....
    # Paste the URL with the token you copied

## 2.0 Assembly of Metatranscriptomes (kept for documentation purposes only)

In [ ]:
# this assembly was done using both noninduced and induced samples
# current analysis has separated these sample types so I will leave all steps below "2.0 Assembly of Metatranscripomtes (kept for documentation purposes only)" for reference
    # and no to be used for this metatranscriptome study

In [ ]:
# transfer data onto the UCM HPC to perform assemblies since the local workstation does not have enough computational resources
# several options:
    # remote transfer via local upload to HPC via VPN connection from laptop
    rsync -avP rrna_r clee289@login.rc.ucmerced.edu
        # use rsync for transferring large files so if connection drops, you can resume from where it left
    # remote transfer from workstation to HPC via VPN connection
    # I used both ways and all files are now up on the HPC borgstorage

# QC'ed rRNA removed reads are stored here: /home/clee289/borgstore/HVBMT_analysis/temp/rrna_r
# upload metadata_ID.txt to borgstorage

scp HVBMG_metadata_combined.txt clee289@login.rc.ucmerced.edu:/home/clee289/borgstore/

In [ ]:
# Assembly with Megahit
##############################
############ NOTE ############
##############################
    # YOU MUST RUN THIS JOB IN THE BORGSTORE DIR
    # submit job to node with high memory


#!/bin/bash
#SBATCH --nodes=1
#SBATCH --partition=cenvalarc.bigmem
#SBATCH --mem=0
#SBATCH --time=3-00:00:00
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=64
#SBATCH --output=rna_assembly_test.log
#SBATCH --job-name=rna_assembly
#SBATCH --mail-user=clee289@ucmerced.edu
#SBATCH --mail-type=ALL
#SBATCH --export=ALL
##SBATCH --constraint=ib

source activate megahit

time megahit -t 64 \
    -1 `tail -n+2 HVBMT_metadata_combined_updated.csv | cut -f1 -d',' | sed 's/^/rrna_r\//;s/$/_1.nonrrna.fastq/'|tr '\n' ','|sed 's/,$//'` \
    -2 `tail -n+2 HVBMT_metadata_combined_updated.csv | cut -f1 -d',' | sed 's/^/rrna_r\//;s/$/_2.nonrrna.fastq/'|tr '\n' ','|sed 's/,$//'` \
    -o megahit

# took ~ 25 hours (1 day)


In [ ]:
# assess assembly quality
quast.py final.contigs.fa \
  -o /mnt/hdd2/clee/transcriptome/result/quast \
  -t 32 -m 1

## 2.1 Read mapping QC'ed reads to Cross assembled Metatranscriptome

In [ ]:
# this part was done with the co-assembled sample assembly (i.e. all noninduced and induced samples combined)

### work in this cell is performed on the UCM HPC

In [ ]:
#!/bin/bash
#SBATCH --nodes=1
#SBATCH --partition=cenvalarc.bigmem
#SBATCH --mem=0
#SBATCH --time=3-00:00:00
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=64
#SBATCH --output=transcriptome_salmon.log
#SBATCH --job-name=rna_assembly
#SBATCH --mail-user=clee289@ucmerced.edu
#SBATCH --mail-type=ALL
#SBATCH --export=ALL

# conda env that contains salmon
source activate megahit

# index megahit cross assembly
salmon index -t /home/clee289/borgstore/HVBMT_analysis/result/megahit/final.contigs.fa \
  -p 64 -i /home/clee289/borgstore/HVBMT_analysis/temp/salmon/megahit_index

# Run multiple quantification tasks in parallel:
# Using rush to run 2 tasks simultaneously.

time tail -n+2 /home/clee289/borgstore/HVBMT_analysis/HVBMT_metadata_combined_updated.csv | cut -f1 -d',' | rush -j 2 \
  "salmon quant -i /home/clee289/borgstore/HVBMT_analysis/temp/salmon/megahit_index -l A -p 32 --meta \
    -1 /home/clee289/borgstore/HVBMT_analysis/temp/rrna_r/{1}_1.nonrrna.fastq \
    -2 /home/clee289/borgstore/HVBMT_analysis/temp/rrna_r/{1}_2.nonrrna.fastq \
    -o /home/clee289/borgstore/HVBMT_analysis/temp/salmon/salmon_mapping/{1}.quant --allowDovetail"


# Merge results:
# make path for new directory to store results
mkdir -p /home/clee289/borgstore/HVBMT_analysis/result/salmon
# merge the TPM scores for all samples
salmon quantmerge --quants /home/clee289/borgstore/HVBMT_analysis/temp/salmon/salmon_mapping/*.quant \
    -o /home/clee289/borgstore/HVBMT_analysis/result/salmon/mapped_reads.TPM
# merge the raw counts for all samples
salmon quantmerge --quants /home/clee289/borgstore/HVBMT_analysis/temp/salmon/salmon_mapping/*.quant \
    --column NumReads -o /home/clee289/borgstore/HVBMT_analysis/result/salmon/mapped_reads.count
# remove the ".quant" extension from sample names in the final output files 
sed -i '1 s/.quant//g' /home/clee289/borgstore/HVBMT_analysis/result/salmon/mapped_reads.*

# Preview the result tables:
# Display the first 3 lines of the quantification results.
head -n3 /home/clee289/borgstore/HVBMT_analysis/result/salmon/mapped_reads.*

# for quick check on mapping rate stats:
# cd /home/clee/project/meta/temp/salmon/map_to_megahit_cross_assembly
grep 'Mapping rate' /home/clee289/borgstore/HVBMT_analysis/temp/salmon/salmon_mapping/HVBMT??.quant/logs/salmon_quant.log \
> /home/clee289/borgstore/HVBMT_analysis/result/salmon/megahit_maprate.txt




## 2.2 Read mapping QC'ed reads to NRCLIB from HVVMG Metagenome

In [ ]:
# this part was done with the co-assembled sample assembly (i.e. all noninduced and induced samples combined)

In [ ]:
# Map cDNA to contigs_nrclib.fasta
# Located here: /home/immunoviromics/HVVMG_crass_ss_combined_5kb_vpf_output/contiglib/contigs_nrclib.fasta (2384 contigs)
    # This determines which contigs are most actively expressed and I know their taxa from MMSeqs2
# Map cDNA to viral NRCLIB NR gene/proteins to determine most actively expressed proteins and their KEGG pathways
    # use the gene list since the cDNA reads can't be mapped to the protein sequences
# Located here: /home/immunoviromics/HVVMG_crass_ss_combined_5kb_vpf_output/nrprot/prot_rep_seq.fasta
# Located here: /home/immunoviromics/HVVMG_crass_ss_combined_5kb_vpf_output/nrprot/prot_rep_seq.fasta

############################### I want to determine the number of proteins encoded by each viral contig (BEGIN) ###############################
# first make a list of each ~2400 viral contig
cd /home/immunoviromics/HVVMG_crass_ss_combined_5kb_vpf_output/contiglib
grep '>' contigs_nrclib.fasta > all_contig_names.txt
# remove thte '>' fasta header
sed 's/^>//' all_contig_names.txt > clean_contig_names.txt
rm all_contig_names.txt
mv clean_contig_names.txt all_contig_names.txt

# create a file to record the outputs
cd /home/immunoviromics/HVVMG_crass_ss_combined_5kb_vpf_output/nrprot
echo "contig_name prot_count" > vir_contig_prot_count.txt
# run loop
while read -r i; do
    prot_count=$(grep -c "$i" prot_rep_seq.fasta)
    echo "$i $prot_count" >> vir_contig_prot_count.txt
done < /home/immunoviromics/HVVMG_crass_ss_combined_5kb_vpf_output/contiglib/all_contig_names.txt
# check the # of viral contigs with no proteins
grep -wc '0' vir_contig_prot_count.txt # OUTPUT = 12 contigs
#NODE_56_length_41588_cov_54.807647_56 0
#NODE_84_length_37174_cov_100.703440_84 0
#NODE_596_length_12330_cov_26.662403_596 0
#NODE_1735_length_6159_cov_44.061435_1735 0
#NODE_4_length_7943_cov_22.159736_3118868 0
#NODE_1_length_381858_cov_23.289228_3539813 0
#NODE_6_length_45650_cov_35.457221_3598097 0
#NODE_10_length_9012_cov_19.762309_3654114 0
#NODE_19_length_5408_cov_22.330095_4700210 0
#NODE_5_length_44967_cov_153.931911_4958566 0
#NODE_7_length_43613_cov_134.973782_4958568 0
#NODE_7_length_28936_cov_10596.145528_6003954 0

# checking the cluster file shows that these viral contigs share proteins with other viral contigs
grep -c NODE_56_length_41588_cov_54.807647_56 prot_cluster.tsv # OUTPUT 65
grep -c NODE_7_length_28936_cov_10596.145528_6003954 prot_cluster.tsv # OUTPUT 42

############################### I want to determine the number of proteins encoded by each viral contig (END) ###############################

############################### I need to filter the viral nr gene list with the nr protein list for metatranscriptome (BEGIN) ###############################
# extract nr protein names
conda activate seqkit
seqkit fx2tab -n -l prot_rep_seq.fasta | cut -f1-9 -d' ' > hvvmg_nrprot_names.txt
seqkit fx2tab -n -l prot_rep_seq.fasta | cut -f1 -d' ' > hvvmg_nrprot_firstnames.txt
# quick check
# (seqkit) immunoviromics@immunoviromics-Z10PE-D16-WS:~/HVVMG_crass_ss_combined_5kb_vpf_output/nrprot$ wc -l hvvmg_nrprot_names.txt 
# 30521 hvvmg_nrprot_names.txt

# move hvvmg_nrprot_names.txt into nrgene directory

seqkit grep -n -f hvvmg_nrprot_names.txt gene_rep_seq.fasta > filtered_gene_rep_seq.fasta
seqkit grep -f hvvmg_nrprot_firstnames.txt gene_rep_seq.fasta > filtered_gene_rep_seq.fasta

grep -c '>' gene_rep_seq.fasta 
31053
grep -c '>' filtered_gene_rep_seq.fasta
30466
wc -l hvvmg_nrprot_firstnames.txt
30521 hvvmg_nrprot_firstnames.txt

# get a clean list of the headers that were successfully filtered into your new file
grep '>' filtered_gene_rep_seq.fasta | sed 's/>//' | awk '{print $1}' > found_names.txt

# Compare the two lists
grep -Fvx -f found_names.txt hvvmg_nrprot_firstnames.txt > missing_names.txt
# the names listed in missing_names.txt are clustered to another sequence (clustered pairs are defined in gene_cluster.tsv)
# However, some of those clustered gene sequences (w/in gene_cluster.tsv) do not have corresponding protein sequences
# therefore, use the KEGG annotations provided by the cluster pair for both of them

#Instead of trying to figure out which specific member a gene sequence came from, treat the cluster itself as the unit of expression. 
    # Map reads only to gene_rep_seq.fasta.
    # Generate a count table where the "gene ID" is the Representative (e.g., NODE_24...).
    # Map metadata back: When you get your results, you use your gene_cluster.tsv to say: "This expression level represents the entire cluster, which includes NODE_24 and NODE_190."

############################### I need to filter the viral nr gene list with the nr protein list for metatranscriptome (END) ###############################

# I am confident I can try mapping the qc'ed metatranscriptome reads to both the nrclib viral contigs and the nr gene list
# I will just have to make sure that those viral contigs listed with 0 proteins are correctly matched with their cluster

Files
HVVMG nrprot 
/home/clee289/borgstore/HVVMG/temp/hvvmg_nrclib_nrprot/prot_rep_seq.fasta
HVVMG NRCLIB 
/home/clee289/borgstore/HVVMG/temp/hvvmg_nrclib_nrprot/contigs_nrclib.fasta
#!/bin/bash
#SBATCH --nodes=1
#SBATCH --partition=long
#SBATCH --mem=0
#SBATCH --time=3-00:00:00
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=56
#SBATCH --output=hvvmg_salmon.log
#SBATCH --job-name=salmon
#SBATCH --mail-user=clee289@ucmerced.edu
#SBATCH --mail-type=ALL
#SBATCH --export=ALL

# conda env that contains salmon
source activate megahit

### HVVMG NRCLIB
# index HVVMG NRCLIB
salmon index -t /home/clee289/borgstore/HVVMG/temp/hvvmg_nrclib_nrprot/contigs_nrclib.fasta \
  -p 56 -i /home/clee289/borgstore/HVVMG/temp/salmon_nrclib/contig_nrclib_index

# Run multiple quantification tasks in parallel:
# Using rush to run 2 tasks simultaneously.

time tail -n+2 /home/clee289/borgstore/HVBMT_analysis/HVBMT_metadata_combined_updated.csv | cut -f1 -d',' | rush -j 2 \
  "salmon quant -i /home/clee289/borgstore/HVVMG/temp/salmon_nrclib/contig_nrclib_index -l A -p 28 --meta \
    -1 /home/clee289/borgstore/HVBMT_analysis/temp/rrna_r/{1}_1.nonrrna.fastq \
    -2 /home/clee289/borgstore/HVBMT_analysis/temp/rrna_r/{1}_2.nonrrna.fastq \
    -o /home/clee289/borgstore/HVVMG/temp/salmon_nrclib_mapping/{1}.quant --allowDovetail"

# Merge results:
# make path for new directory to store results
mkdir -p /home/clee289/borgstore/HVVMG/result/salmon
# merge the TPM scores for all samples
salmon quantmerge --quants /home/clee289/borgstore/HVVMG/temp/salmon_nrclib_mapping/*.quant \
    -o /home/clee289/borgstore/HVVMG/result/salmon/mapped_reads.TPM
# merge the raw counts for all samples
salmon quantmerge --quants /home/clee289/borgstore/HVVMG/temp/salmon_nrclib_mapping/*.quant \
    --column NumReads -o /home/clee289/borgstore/HVVMG/result/salmon/mapped_reads.count
# remove the ".quant" extension from sample names in the final output files 
sed -i '1 s/.quant//g' /home/clee289/borgstore/HVVMG/result/salmon/mapped_reads.*

# for quick check on mapping rate stats:
# cd /home/clee/project/meta/temp/salmon/map_to_megahit_cross_assembly
grep 'Mapping rate' /home/clee289/borgstore/HVVMG/temp/salmon_nrclib_mapping/HVBMT??.quant/logs/salmon_quant.log \
> /home/clee289/borgstore/HVVMG/result/salmon/maprate.txt

#!/bin/bash
#SBATCH --nodes=1
#SBATCH --partition=long
#SBATCH --mem=0
#SBATCH --time=3-00:00:00
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=56
#SBATCH --output=hvvmg_nrgene_salmon.log
#SBATCH --job-name=shvvmg_nrgene_salmon
#SBATCH --mail-user=clee289@ucmerced.edu
#SBATCH --mail-type=ALL
#SBATCH --export=ALL

### HVVMG NRGENE
# conda env that contains salmon
source activate megahit

# index HVVMG NRGENE
salmon index -t /home/clee289/borgstore/HVVMG/temp/hvvmg_nrclib_nrprot/gene_rep_seq.fasta \
  -p 56 -i /home/clee289/borgstore/HVVMG/temp/salmon_nrgene/nrgene_index

# Run multiple quantification tasks in parallel:
# Using rush to run 2 tasks simultaneously.

time tail -n+2 /home/clee289/borgstore/HVBMT_analysis/HVBMT_metadata_combined_updated.csv | cut -f1 -d',' | rush -j 2 \
  "salmon quant -i /home/clee289/borgstore/HVVMG/temp/salmon_nrgene/nrgene_index -l A -p 28 --meta \
    -1 /home/clee289/borgstore/HVBMT_analysis/temp/rrna_r/{1}_1.nonrrna.fastq \
    -2 /home/clee289/borgstore/HVBMT_analysis/temp/rrna_r/{1}_2.nonrrna.fastq \
    -o /home/clee289/borgstore/HVVMG/temp/salmon_nrgene_mapping/{1}.quant --allowDovetail"

# Merge results:
# make path for new directory to store results
# mkdir -p /home/clee289/borgstore/HVVMG/result/salmon
# merge the TPM scores for all samples
salmon quantmerge --quants /home/clee289/borgstore/HVVMG/temp/salmon_nrgene_mapping/*.quant \
    -o /home/clee289/borgstore/HVVMG/result/salmon/nrgene_mapped_reads.TPM
# merge the raw counts for all samples
salmon quantmerge --quants /home/clee289/borgstore/HVVMG/temp/salmon_nrgene_mapping/*.quant \
    --column NumReads -o /home/clee289/borgstore/HVVMG/result/salmon/nrgene_mapped_reads.count
# remove the ".quant" extension from sample names in the final output files 
sed -i '1 s/.quant//g' /home/clee289/borgstore/HVVMG/result/salmon/nrgene_mapped_reads.*

# for quick check on mapping rate stats:
# cd /home/clee/project/meta/temp/salmon/map_to_megahit_cross_assembly
grep 'Mapping rate' /home/clee289/borgstore/HVVMG/temp/salmon_nrgene_mapping/HVBMT??.quant/logs/salmon_quant.log \
> /home/clee289/borgstore/HVVMG/result/salmon/nrgene_maprate.txt





## 2.3 Read mapping QC'ed reads to NRCLIB and NR Proteins from HVBMG Metagenome

In [ ]:
# this part was done with the co-assembled sample assembly (i.e. all noninduced and induced samples combined)

In [ ]:
# use this file for nr proteins
/home/clee289/data/kraken2_work-dir/nucleotide.renamed.fa

#Files
#HVBMG nrprot 

/home/clee289/borgstore/HVBMG/temp/hvbmg_nrclib_nrprot/prot_rep_seq.fasta

#HVBMG NRCLIB 
# on the local workstation:
/home/clee/project/meta/result/nrclib/contigs_nrclib.fasta
# upload onto the UCM HPC here
scp -r contigs_nrclib.renamed.fasta clee289@login.rc.ucmerced.edu:/home/clee289/borgstore/HVBMG/temp/hvbmg_nrclib_nrprot
    # located here now:
    /home/clee289/borgstore/HVBMG/temp/hvbmg_nrclib_nrprot/contigs_nrclib.fasta

#!/bin/bash
#SBATCH --nodes=1
#SBATCH --partition=cenvalarc.bigmem
#SBATCH --mem=0
#SBATCH --time=3-00:00:00
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=64
#SBATCH --output=hvbmg_salmon.log
#SBATCH --job-name=salmon
#SBATCH --mail-user=clee289@ucmerced.edu
#SBATCH --mail-type=ALL
#SBATCH --export=ALL

# conda env that contains salmon
source activate megahit

### HVBMG NRCLIB
# index HVBMG NRCLIB
salmon index -t /home/clee289/borgstore/HVBMG/temp/hvbmg_nrclib_nrprot/contigs_nrclib.renamed.fasta \
  -p 64 -i /home/clee289/borgstore/HVBMG/temp/salmon_nrclib/contig_nrclib_index

# Run multiple quantification tasks in parallel:
# Using rush to run 2 tasks simultaneously.

time tail -n+2 /home/clee289/borgstore/HVBMT_analysis/HVBMT_metadata_combined_updated.csv | cut -f1 -d',' | rush -j 2 \
  "salmon quant -i /home/clee289/borgstore/HVBMG/temp/salmon_nrclib/contig_nrclib_index -l A -p 32 --meta \
    -1 /home/clee289/borgstore/HVBMT_analysis/temp/rrna_r/{1}_1.nonrrna.fastq \
    -2 /home/clee289/borgstore/HVBMT_analysis/temp/rrna_r/{1}_2.nonrrna.fastq \
    -o /home/clee289/borgstore/HVBMG/temp/salmon_nrclib_mapping/{1}.quant --allowDovetail"

# Merge results:
# make path for new directory to store results
#mkdir -p /home/clee289/borgstore/HVBMG/result/salmon
# merge the TPM scores for all samples
salmon quantmerge --quants /home/clee289/borgstore/HVBMG/temp/salmon_nrclib_mapping/*.quant \
    -o /home/clee289/borgstore/HVBMG/result/salmon/mapped_reads.TPM
# merge the raw counts for all samples
salmon quantmerge --quants /home/clee289/borgstore/HVBMG/temp/salmon_nrclib_mapping/*.quant \
    --column NumReads -o /home/clee289/borgstore/HVBMG/result/salmon/mapped_reads.count
# remove the ".quant" extension from sample names in the final output files 
sed -i '1 s/.quant//g' /home/clee289/borgstore/HVBMG/result/salmon/mapped_reads.*

# for quick check on mapping rate stats:
# cd /home/clee/project/meta/temp/salmon/map_to_megahit_cross_assembly
grep 'Mapping rate' /home/clee289/borgstore/HVBMG/temp/salmon_nrclib_mapping/HVBMT??.quant/logs/salmon_quant.log \
> /home/clee289/borgstore/HVBMG/result/salmon/maprate.txt

#!/bin/bash
#SBATCH --nodes=1
#SBATCH --partition=long
#SBATCH --mem=0
#SBATCH --time=3-00:00:00
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=56
#SBATCH --output=hvvmg_nrgene_salmon.log
#SBATCH --job-name=shvvmg_nrgene_salmon
#SBATCH --mail-user=clee289@ucmerced.edu
#SBATCH --mail-type=ALL
#SBATCH --export=ALL

### HVBMG NRGENE
# conda env that contains salmon
source activate megahit

# index HVBMG NRGENE
salmon index -t /home/clee289/borgstore/HVBMG/temp/hvbmg_nrclib_nrprot/nucleotide.renamed.fa \
  -p 56 -i /home/clee289/borgstore/HVBMG/temp/salmon_nrgene/nrgene_index

# Run multiple quantification tasks in parallel:
# Using rush to run 2 tasks simultaneously.

time tail -n+2 /home/clee289/borgstore/HVBMT_analysis/HVBMT_metadata_combined_updated.csv | cut -f1 -d',' | rush -j 2 \
  "salmon quant -i /home/clee289/borgstore/HVBMG/temp/salmon_nrgene/nrgene_index -l A -p 28 --meta \
    -1 /home/clee289/borgstore/HVBMT_analysis/temp/rrna_r/{1}_1.nonrrna.fastq \
    -2 /home/clee289/borgstore/HVBMT_analysis/temp/rrna_r/{1}_2.nonrrna.fastq \
    -o /home/clee289/borgstore/HVBMG/temp/salmon_nrgene_mapping/{1}.quant --allowDovetail"

# Merge results:
# make path for new directory to store results
# mkdir -p /home/clee289/borgstore/HVBMG/result/salmon
# merge the TPM scores for all samples
salmon quantmerge --quants /home/clee289/borgstore/HVBMG/temp/salmon_nrgene_mapping/*.quant \
    -o /home/clee289/borgstore/HVBMG/result/salmon/nrgene_mapped_reads.TPM
# merge the raw counts for all samples
salmon quantmerge --quants /home/clee289/borgstore/HVBMG/temp/salmon_nrgene_mapping/*.quant \
    --column NumReads -o /home/clee289/borgstore/HVBMG/result/salmon/nrgene_mapped_reads.count
# remove the ".quant" extension from sample names in the final output files 
sed -i '1 s/.quant//g' /home/clee289/borgstore/HVBMG/result/salmon/nrgene_mapped_reads.*

# for quick check on mapping rate stats:
# cd /home/clee/project/meta/temp/salmon/map_to_megahit_cross_assembly
grep 'Mapping rate' /home/clee289/borgstore/HVBMG/temp/salmon_nrgene_mapping/HVBMT??.quant/logs/salmon_quant.log \
> /home/clee289/borgstore/HVBMG/result/salmon/nrgene_maprate.txt

# 3.0 Annotation/Gene calling on Metranscriptome Assembly

## This entire section of 3.0 Annotation/Gene calling on Metatranscriptome Assembly is on indefinite pause. These sections will just be kept here for documentation purposes only.
## Reason for halting this analysis is that after much deliberation, I think the best use of my time would be to utilize the viral and bacterial metagenomes for downstream metatranscriptomic analysis. The two reasons being (1) predicting ORFs from metatranscriptomic data is viable but not optimal since bacterial transcripts can be fragmented and incomplete which leads to an incomplete gene database for annotation and anaylsis and (2) I care about community-level differences in expressions, not strain-level.

## For those case reasons described above a metagenome covers everything since it can generate a more complete gene database and provides the static background noise which allows for more accurate mapping to those coding regions. If a metagenome was unavailable, then a ORF prediction from a metranscriptoome may seem more plausible.

## 3.1 Determining ORF sequence threshold for Transdecoder

In [ ]:
# I want to get the statistics of the nonredundant genes list from the metagenomes to know how long the average protein sequence is
# Transdecoder defaults to searches for ORFs with a min length of 100 amino acids
# I am worried that it may remove a lot of proteins from the metagenomes previously called by prodigal

# run this to get quick stats on nr proteins from bacterial metagenomes
seqkit stats protein.renamed.fa 
# output 
#(seqkit) immunoviromics@immunoviromics-Z10PE-D16-WS:/home/clee/project/meta/result/NR/nrclib_NR$ seqkit stats protein.renamed.fa 
#file                format  type      num_seqs      sum_len  min_len  avg_len  max_len
#protein.renamed.fa  FASTA   Protein  1,784,945  222,632,372        0    124.7   13,317

# to extract the length of each sequence into a text file for quickly plotting
seqkit fx2tab -n -l protein.renamed.fa > hvbmg_nr_protein_length.txt

# import the text file into R for quick plot and statistics
# total number of protein sequences: 1,784,945
# total number of protein sequences > 100 : 672,335 (`38%)

# filtering for counts of proteins >100 showed that most proteins in the HVBMG NR proteins were <100 aa in sequence length
# I will set the Transdecoder threshold to 75 to account for viral sORFs that may be <100 aa in length (i.e. viroporins)

## 3.2 Running Transdecoder to predict ORFS >75 AA seq length

In [ ]:
# ran on the UCM HPC 
#!/bin/bash
#SBATCH --nodes=1
#SBATCH --partition=bigmem
#SBATCH --mem=0
#SBATCH --time=3-00:00:00
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=56
#SBATCH --output=transdecoder.log
#SBATCH --job-name=transdecoder
#SBATCH --mail-user=clee289@ucmerced.edu
#SBATCH --mail-type=ALL
#SBATCH --export=ALL

source activate transdecoder

#mkdir -p /home/clee289/borgstore/HVBMT_analysis/temp/transdecoder
#cp result/megahit/final.contigs.fa temp/transdecoder/

TransDecoder.LongOrfs -t final.contigs.fa \
-m 75 \
--output_dir /home/clee289/borgstore/HVBMT_analysis/temp/transdecoder

## 3.3 Generating NR proteins in Metatranscriptome Assembly

### I'm putting this on hold for now since clustering can lead to loss of strain-specific differences between samples. I moved on to pass the ~1.1 million predicted ORFs from Transdecoder to Salmon for read mapping in section 3.4

In [ ]:
# use CD-HIT to perform Clustering / Redundancy Removal
# the predicted ORFs from Transdecoder will be clustered together to generate a NR library of genes present in the metatranscriptome assembly

# Input file: gene sequences predicted by Transdecoder → longest_orfs.cds
    # path on UCM HPC: /home/clee289/borgstore/HVBMT_analysis/temp/transdecoder/final.contigs.fa.transdecoder_dir/longest_orfs.cds

# Output files: non-redundant gene and protein sequences →
# result/NR/nucleotide.fa, result/NR/protein.fa

# location to store results:
mkdir -p /home/clee289/borgstore/HVBMT_analysis/result/NR
# location to store temp files:
mkdir -p /home/clee289/borgstore/HVBMT_analysis/temp/cdhit


#!/bin/bash
#SBATCH --nodes=1
#SBATCH --partition=cenvalarc.bigmem
#SBATCH --mem=0
#SBATCH --time=3-00:00:00
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=64
#SBATCH --output=cdhit.log
#SBATCH --job-name=cdhit
#SBATCH --mail-user=clee289@ucmerced.edu
#SBATCH --mail-type=ALL
#SBATCH --export=ALL

source activate megahit

# aS: sequence coverage threshold
# c: sequence identity threshold
# G: local alignment mode
# g: best clustering option
# T: number of threads
# M: memory limit in Mb (0 = no limit) 
# Example: for 20,000 genes requires ~2 minutes; for 3 million genes, 384 threads, ~15 minutes; 
# for 20 million genes, ~2000 hours — multi-threading can speed it up.
cd-hit-est -i /home/clee289/borgstore/HVBMT_analysis/temp/transdecoder/final.contigs.fa.transdecoder_dir/longest_orfs.cds \
    -o /home/clee289/borgstore/HVBMT_analysis/result/NR/nucleotide.fa \
    -aS 0.9 -c 0.95 -G 0 -g 0 -T 64 -M 0

source deactivate
source activate seqkit

# Count the number of non-redundant genes.
# The reduction per single assembly is usually small (e.g., from 3M → 2M),
# but redundancy is higher when combining multiple batches.
grep -c '>' /home/clee289/borgstore/HVBMT_analysis/result/NR/nucleotide.fa > /home/clee289/borgstore/HVBMT_analysis/result/NR/NR_gene_count.txt
seqkit stats /home/clee289/borgstore/HVBMT_analysis/result/NR/nucleotide.fa > /home/clee289/borgstore/HVBMT_analysis/result/NR/stats.txt

############################################# STOPPED HERE (DID NOT RUN ANYTHING BELOW THIS LINE IN THIS CELL ##################################################


########################### EXTREMELY IMPORTANT BEGIN ###########################
# fix headers of nucleotide.fa for later input into salmon and the corresponding protein.fa into eggnog so that their gene names will match
# this will allow me to identify what each gene called by prodigal is (from eggnog's protein databases) and tie them with relative abundance (from salmon's read mapping)
# therefore it is critical that I use the same gene names to cross reference these results together and prevent confusion downstream

# edits shall be made to nucleotide.fa output file from CD-HIT's clustering on Prodigals gene.fa file

# first Prodigal creates extra information in sequence headers of the nucleotide.fa file we don't need (original file will be kept in just in case)
# delete everything after the first space in the sequence headers
sed '/^>/s/ .*//' nucleotide.fa > nucleotide.nospace.fa

# then check that no duplicate names exist for each sequence header since salmon cannot build a database with duplicate sequence names
grep '^>' nucleotide.nospace.fa | sed 's/>//' | cut -d' ' -f1 | sort | uniq -d | wc -l
# returns 54 sequence headers
# count number of duplicate for each sequence
grep '^>' nucleotide.nospace.fa | sed 's/>//' | cut -d' ' -f1 | sort | uniq -c | awk '$1 > 1'

# to make a deduplicated nucleotide file with no duplicate headers use:
awk '
  /^>/ {
    id = substr($0, 2)       # remove leading >
    count[id]++              # increment duplicate counter

    if (count[id] == 1) {
      print ">" id
    } else {
      print ">" id "_dup" count[id]
    }
    next
  }
  { print }
' nucleotide.nospace.fa > nucleotide.renamed.fa

# then quick check (both should return nothing)
grep '^>' nucleotide.renamed.fa | sed 's/>//' | cut -d' ' -f1 | sort | uniq -d | wc -l
grep '^>' nucleotide.renamed.fa | sed 's/>//' | cut -d' ' -f1 | sort | uniq -c | awk '$1 > 1'

# check that dups are named as such and the number stays the same
grep -c 'dup' nucleotide.renamed.fa
# returns 54 sequences with dups that is correct since its the same as before

# Now that I have cleaned the sequence names and ensure there are no duplicate sequence names
# I can translate this clean nucleotide sequences to corresponding protein sequences for use in eggnog annotation
# --trim removes trailing '*' characters.
seqkit translate --trim nucleotide.renamed.fa \
    > protein.renamed.fa

### USE THESE TWO FILES FOR DOWNSTREAM
# directory for files is here: /home/clee/project/meta//home/clee289/borgstore/HVBMT_analysis/result/NR
# nucleotide.renamed.fa for salmon gene quantification
# protein.renamed.fa for protein annotation with eggnog databases

########################### EXTREMELY IMPORTANT END ###########################

# We shall keep an modified protein.fa file as well just in case
# Translate nucleotide sequences to corresponding protein sequences.
# --trim removes trailing '*' characters.
seqkit translate --trim /home/clee289/borgstore/HVBMT_analysis/result/NR/nucleotide.fa \
    > /home/clee289/borgstore/HVBMT_analysis/result/NR/protein.fa

# Print the header with tab-separated fields
head -n1 /home/clee289/borgstore/HVBMT_analysis/result/NR/stats.txt | tr -s ' ' '\t' > result/NR/all_stats.txt

# Loop through folders and append second line with tab separation
for folder in result/NR/*/; do
    # Check if the folder contains a stats.txt file
    if [ -f "${folder}/stats.txt" ]; then
        # Extract the second line of stats.txt
        second_line=$(sed -n '2p' "${folder}/stats.txt")
        
        # Normalize the spaces and convert to tabs
        second_line=$(echo "$second_line" | tr -s ' ' '\t')
        
        # Append the second line to all_stats.txt file with tabs
        echo -e "$second_line" >> result/NR/all_stats.txt
    else
        echo "Warning: stats.txt not found in $folder"
    fi
done


# get output of number of complete genes
for i in /home/clee/project/meta//home/clee289/borgstore/HVBMT_analysis/result/NR; do
    grep -c 'partial=00' ${i}/protein.fa > ${i}/complete_genes.txt
done


# make file for total complete genes for dereplicated genes assemblies ( for each Single-sample assembly + megahit cross assembly + nr contig lib)
cd /home/clee/project/meta/result/NR
echo -e "SampleID\tcomplete_genes" > all_complete_genes.txt

for folder in /home/clee/project/meta/result/NR/*/; do
    # Check if it's a directory and not a file
    if [ -d "$folder" ]; then
        # Extract the folder name (e.g., HVBMG01)
        sample_id=$(basename "$folder")
        
        # Check if the complete_genes.txt file exists in the directory
        if [ -f "${folder}/complete_genes.txt" ]; then
            # Read the complete_genes.txt file inside the folder
            complete_genes=$(cat "${folder}/complete_genes.txt")
            
            # Append the sample_id and complete_genes to the output file
            echo -e "$sample_id\t$complete_genes" >> all_complete_genes.txt
        else
            # Handle the case where the file doesn't exist
            echo "Warning: complete_genes.txt not found in $folder"
        fi
    fi
done

## 3.4 Read mapping predicted ORFs with Salmon

In [ ]:
# generate a decoy-aware transcriptome to mitigate spurious mappings to noncoding regions

# my target mappings are the output predicted ORFs from Transdecoder:
# targets.fa = transdecoder.cds

# make working directory
mkdir -p /home/clee289/borgstore/HVBMT_analysis/temp/salmon/salmon_mapping_decoy

# generate the decoy list by extracting the names of transcript assembly contigs:
grep "^>" /home/clee289/borgstore/HVBMT_analysis/result/megahit/final.contigs.fa | sed 's/>//' > /home/clee289/borgstore/HVBMT_analysis/temp/salmon/salmon_mapping_decoy/decoys.txt

# then concatenate targets & decoys
# Salmon requires a single FASTA containing both my target (the ORFs) and the transcript assembly contig
cat /home/clee289/borgstore/HVBMT_analysis/temp/transdecoder/final.contigs.fa.transdecoder_dir/longest_orfs.cds \
/home/clee289/borgstore/HVBMT_analysis/result/megahit/final.contigs.fa \
> /home/clee289/borgstore/HVBMT_analysis/temp/salmon/salmon_mapping_decoy/gentrome.fa

# build the decoy-aware index
salmon index \
  -t gentrome.fa \
  -d decoys.txt \
  -i salmon_index \
  --gencode
